<a href="https://colab.research.google.com/github/aminrafi394/ai-second-project/blob/Q2_Task2/Section2_Part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reading Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:

from google.colab import drive


drive.mount('/content/drive')
path_train = '/content/drive/MyDrive/train_data.csv'
train_data = pd.read_csv(path_train)

path_test = '/content/drive/MyDrive/test_data.csv'
test_data = pd.read_csv(path_test)


Mounted at /content/drive


/tmp/ipykernel_1734/3746809614.py:6: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  train_data = pd.read_csv(path_train)


# Preproccesing

**کد های پیش پردازش داده شده به منظور تمیز کردن داده ها  
تنها تغیرات ایحاد شده روی کد تغیر مسیر ها و برخی کد ها برای همخوانی و احرا شدن توسط فایل های درون google drive بود**

In [ ]:
from pathlib import Path
import ast
import html
import re
import unicodedata

from google.colab import drive
import pandas as pd

drive.mount("/content/drive")

PATH_TRAIN = Path("/content/drive/MyDrive/train_data.csv")
PATH_TEST = Path("/content/drive/MyDrive/test_data.csv")
PATH_PRODUCTS = Path("/content/drive/MyDrive/title_brand.csv")

OUTPUT_TRAIN = Path("/content/drive/MyDrive/train_sentiment_ready.csv")
OUTPUT_TEST = Path("/content/drive/MyDrive/test_sentiment_ready.csv")


def normalize_asin(values: pd.Series) -> pd.Series:
    """یکسان‌سازی شناسه محصول به حروف بزرگ و حذف فاصله‌های اضافی."""
    return values.astype("string").str.strip().str.upper()


def clean_text(value) -> str:
    """پاک‌سازی کاراکترهای نامعتبر، کدهای HTML و آدرس‌ها بدون حذف علائم حسی."""
    if value is None or pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKC", str(value))
    text = html.unescape(text)
    text = re.sub(r"(?i)https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"(?is)<(script|style).*?>.*?</\1>", " ", text)
    text = re.sub(r"(?s)<[^>]+>", " ", text)
    text = re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def format_style(value) -> str:
    """تبدیل رشته‌های ساختار دیکشنری فیلد style به متن منظم."""
    if value is None or pd.isna(value) or not str(value).strip():
        return ""
    raw = str(value).strip()
    try:
        parsed = ast.literal_eval(raw)
        if isinstance(parsed, dict):
            parts = []
            for key, item in parsed.items():
                k = str(key).strip().rstrip(":")
                v = clean_text(item)
                if v:
                    parts.append(f"{k}: {v}" if k else v)
            return "; ".join(parts)
    except (ValueError, SyntaxError):
        pass
    return clean_text(raw)


def combine_text(row) -> str:
    """ترکیب خلاصه و متن اصلی بررسی در یک فیلد یکپارچه."""
    summary = row["summary"]
    body = row["reviewText"]
    if summary and body:
        return f"{summary}. {body}"
    return summary or body



print("در حال خواندن فایل محصولات...")
products_raw = pd.read_csv(
    PATH_PRODUCTS,
    encoding="utf-8-sig",
    dtype={"asin": "string"},
    low_memory=False,
)

required_product_columns = {"asin", "title", "brand"}
missing_product_cols = required_product_columns - set(products_raw.columns)
if missing_product_cols:
    raise ValueError(f"ستون‌های مفقود در فایل محصولات: {missing_product_cols}")

products = products_raw.copy()
products["asin_key"] = normalize_asin(products["asin"])
products["title"] = products["title"].astype("string").str.strip()
products["brand"] = products["brand"].astype("string").str.strip()

product_lookup = products.drop_duplicates(subset=["asin_key"])[
    ["asin_key", "title", "brand"]
]

def process_reviews(file_path: Path, is_train: bool = True) -> pd.DataFrame:
    print(f"\nشروع پردازش: {file_path.name}")
    df = pd.read_csv(
        file_path,
        encoding="utf-8-sig",
        dtype={"asin": "string", "reviewerID": "string"},
        low_memory=False,
    )

    base_required = {"asin", "reviewText", "summary", "reviewTime", "verified", "vote", "style"}
    missing = base_required - set(df.columns)
    if missing:
        raise ValueError(f"ستون‌های مفقود در {file_path.name}: {missing}")

    initial_len = len(df)

    df = df.drop_duplicates().copy()
    print(f"ردیف‌های تکراری حذف‌شده: {initial_len - len(df):,}")

    df["asin_key"] = normalize_asin(df["asin"])


    df["reviewText"] = df["reviewText"].map(clean_text)
    df["summary"] = df["summary"].map(clean_text)
    df["style"] = df["style"].map(format_style)


    df["text_for_model"] = df.apply(combine_text, axis=1)


    df["reviewTime"] = pd.to_datetime(df["reviewTime"], errors="coerce").dt.strftime("%Y-%m-%d")
    df["vote"] = pd.to_numeric(df["vote"], errors="coerce").astype("Int64")

    verified_map = {True: True, False: False, "true": True, "false": False, "True": True, "False": False}
    df["verified"] = df["verified"].map(verified_map).astype("boolean")

    merged = df.merge(
        product_lookup,
        on="asin_key",
        how="left",
        validate="many_to_one",
    ).drop(columns=["asin_key"])


    empty_text_mask = merged["text_for_model"].str.strip().eq("")
    if empty_text_mask.any():
        print(f"هشدار: تعداد {empty_text_mask.sum():,} ردیف فاقد هرگونه متن در {file_path.name} یافت شد.")

    return merged


train_df = process_reviews(PATH_TRAIN, is_train=True)
test_df = process_reviews(PATH_TEST, is_train=False)

train_df.to_csv(OUTPUT_TRAIN, index=False, encoding="utf-8-sig", na_rep="")
test_df.to_csv(OUTPUT_TEST, index=False, encoding="utf-8-sig", na_rep="")

print("\n--- پایان پردازش و ذخیره‌سازی ---")
print(f"Train Dataset: {train_df.shape[0]:,} سطر | مسیر: {OUTPUT_TRAIN}")
print(f"Test Dataset:  {test_df.shape[0]:,} سطر | مسیر: {OUTPUT_TEST}")
print("ستون‌های خروجی Train:", list(train_df.columns))
print("ستون‌های خروجی Test:", list(test_df.columns))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
در حال خواندن فایل محصولات...

شروع پردازش: train_data.csv
ردیف‌های تکراری حذف‌شده: 8,327

شروع پردازش: test_data.csv
ردیف‌های تکراری حذف‌شده: 3

--- پایان پردازش و ذخیره‌سازی ---
Train Dataset: 830,617 سطر | مسیر: /content/drive/MyDrive/train_sentiment_ready.csv
Test Dataset:  19,997 سطر | مسیر: /content/drive/MyDrive/test_sentiment_ready.csv
ستون‌های خروجی Train: ['overall', 'vote', 'verified', 'reviewTime', 'reviewerID', 'asin', 'style', 'reviewerName', 'reviewText', 'summary', 'unixReviewTime', 'text_for_model', 'title', 'brand']
ستون‌های خروجی Test: ['vote', 'verified', 'reviewTime', 'reviewerID', 'asin', 'style', 'reviewerName', 'reviewText', 'summary', 'unixReviewTime', 'text_for_model', 'title', 'brand']


**در مرحله ی اول  داده های بخش خلاصه نظرات رو به ابتدای متن نظر اضافه می کنیم چون این خلاصه نظرات باعث میشه این چکیده متن در اول بیاد در این حذف توکن ها (بخاظر تعداد زیاد و محدودیتش) داده های مفید گم نشه**

In [ ]:
from sklearn.preprocessing import  FunctionTransformer

def change_review_text(data_frame: pd.DataFrame):
  data_frame["reviewText"]  =data_frame["summary"] + " " + data_frame["reviewText"]
  data_frame =data_frame.drop(columns=["summary"])

  return data_frame

changer =  FunctionTransformer(change_review_text)
train_data = changer.transform(train_data)
test_data = changer.transform(test_data)



In [ ]:
from sklearn.model_selection import train_test_split

train_data_copy = train_data.copy()
train_data_copy = train_data_copy[:60000]

train, validation = train_test_split(train_data_copy, test_size=0.2, shuffle=True)

validation_x = validation.drop(columns=["overall"])
validation_y = validation["overall"]

train_x = train.drop(columns=["overall"])
train_y = train["overall"]

train_x


,vote,verified,reviewTime,reviewerID,asin,style,reviewerName,reviewText,unixReviewTime
13011,NaN,True,2016-05-25,AOHB3TUD9G5LU,B0002QLDLC,{'Model Number:': ' RBC17'},R. Garrett,Excellent quality but price has increased quit...,1464134400
4822,NaN,False,2016-12-20,A3QJU4FEN8PQSZ,B00006B8DA,"{'Capacity:': ' 4 Port', 'Style:': ' VGA'}",Grimmy,"Works, but you may need some new cables Doesn'...",1482192000
32163,3.0,True,2016-02-25,AF30W6VYOTF7H,B000X3KOD2,{'Color:': ' Black'},Catrine,Perfect for my cable box. This was easy to ins...,1456358400
50177,NaN,True,2017-04-26,AL3IW5EQ5MMGQ,B001E25KDK,NaN,R. Franklin.,great condition with a CD-R containing the mos...,1493164800
49456,NaN,True,2017-03-03,A27J05HXT6JA3R,B001DFTGR4,{'Style:': ' FreeVision'},Richard Egan,"Worked for us We recently ""cut the cable"", but...",1488499200
...,...,...,...,...,...,...,...,...,...
58496,NaN,True,2017-03-06,A14BLKN5AUN5LK,B001XZRTFU,{'Color:': ' Clear'},M &amp; J,Still the best IMHO I've tried all of the othe...,1488758400
2316,2.0,True,2017-05-07,AJV6ZL56NXES,B0000512LA,{'Size:': ' 1200W'},Amazon Customer,Works as advertised The unit seems to work wel...,1494115200
7077,NaN,True,2018-03-31,A303CWSS57GY4P,B00009KLAE,NaN,Raymond Van Hook,Nice UV filter for the price! Decent glass and...,1522454400
33338,NaN,True,2016-03-04,A21LUI699KJSO4,B00110GAQQ,"{'Size:': ' Small', 'Color:': ' Black'}",Justin Gulla,"Perfect for me for diabetic supplies, camera g...",1457049600


# Model Artichecture

**از مدل DitilBErt به عنوان transferlearnign استفاده می کنیم به این صورت که لایه های اصلی و بدنه رو که قبلا آموزش دیدن رو برمی داریم و فریز می کنیم و بعدش مدل آخری رو بر میداریم و لایه های کلاس بندی خودمون رو روش قرار می دیم**

در این خط داده های رو برای آموزش آماده می کنیم   
به این صورت که توسط خود فرآین tokenize مدل پایه این ها توکن گذاری می شن و بعددر صورت اینکه این داده ها به حداکثر مقدار توکن نرسیدن با استفاده از ئدینگ این فضا های خالی پر میشه و اگه این توکن ها بیشتر بود از 250 به بعد قطغ میشه   
نکته دیگه این هست که این توکن ها از اول داده خونده میشه یعنی 250 واژه ی اول برداشته می شوند

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import  AdamW
from transformers import DistilBertModel, DistilBertTokenizerFast,  get_linear_schedule_with_warmup

class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer: DistilBertTokenizerFast, max_len: int = 250):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = str(self.texts[index])
        label = self.labels[index]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long)
        }


تو این بخش یه مدل داریم که به این صورت عمل می کنه   
اول وزن های پایه مدل DIstilBErt رو لود می کنه و بعد از یه لایه خطی قرار میدیم به این علت که اولا ئرش از این تعداد بعد به تعداد 5 بعد خیلی جالب نیست و دیلی دوم هم این هست که باعث میشه لمرز های ئیچیده تر رو بتونه تشخیص بده.  
در مرحله بعد یه تابع امتیویشن داریم و بعدش droupout برای جلوگیری از بیشبرازش و در نهایت خود این کلاس بندی خودمون رو داریم  که میاد یه لیستی از امتیاز ها رو به ما بر می گردونه و بعد با اونها این تصمیم گرفته میشه.

In [ ]:
import torch
import torch.nn as nn
from transformers import DistilBertModel, AutoTokenizer
import pandas as pd
from typing import Optional, Union, List

class DistilBertCustomClassifier(nn.Module):
    def __init__(self, num_classes: int, dropout_rate: float = 0.3, model_name: str = "distilbert-base-uncased"):
        super().__init__()

        self.model_name = model_name
        self.distilbert = DistilBertModel.from_pretrained(model_name)
        hidden_size = self.distilbert.config.hidden_size  # 768

        self.pre_classifier = nn.Linear(hidden_size, 256)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(256, num_classes)

        nn.init.kaiming_normal_(self.pre_classifier.weight)
        nn.init.xavier_normal_(self.classifier.weight)

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]

        x = self.pre_classifier(cls_output)
        x = self.activation(x)
        x = self.dropout(x)
        logits = self.classifier(x)
        return logits

    @torch.no_grad()
    def predict(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, return_probs: bool = False):
        self.eval()
        logits = self.forward(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(logits, dim=-1)
        predictions = torch.argmax(probs, dim=-1)

        if return_probs:
            confidences, _ = torch.max(probs, dim=-1)
            return predictions, confidences

        return predictions

    @torch.no_grad()
    def predict_dataframe(
        self,
        df: pd.DataFrame,
        text_column: str,
        batch_size: int = 32,
        max_length: int = 128,
        pred_column_name: str = "prediction",
        prob_column_name: Optional[str] = "confidence",
        inplace: bool = False
    ) -> pd.DataFrame:
        self.eval()
        target_df = df if inplace else df.copy()

        device = next(self.parameters()).device
        texts = target_df[text_column].astype(str).tolist()

        all_preds: List[int] = []
        all_confs: List[float] = []

        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i : i + batch_size]

            encoded = self.tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            input_ids = encoded["input_ids"].to(device)
            attention_mask = encoded["attention_mask"].to(device)

            logits = self.forward(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(logits, dim=-1)

            confs, preds = torch.max(probs, dim=-1)

            all_preds.extend(preds.cpu().numpy().tolist())
            if prob_column_name:
                all_confs.extend(confs.cpu().numpy().tolist())

        target_df[pred_column_name] = all_preds
        if prob_column_name:
            target_df[prob_column_name] = all_confs

        return target_df["prediction"]


در این قسمت مشخصه ها و هایپرپارامتر های مد نظر مدل داده میشن

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_CLASSES = 6
MAX_LEN = 250
BATCH_SIZE = 32
EPOCHS = 2


tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
dataset = TextClassificationDataset(train_x['reviewText'], train_y, tokenizer, max_len=MAX_LEN)
data_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model = DistilBertCustomClassifier(num_classes=NUM_CLASSES)
model.to(device)

optimizer_grouped_parameters = [
    {"params": model.distilbert.parameters(), "lr": 2e-5},
    {"params": model.pre_classifier.parameters(), "lr": 1e-3},
    {"params": model.classifier.parameters(), "lr": 1e-3}
]

optimizer = AdamW(optimizer_grouped_parameters, weight_decay=0.01)
total_steps = len(data_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
criterion = nn.CrossEntropyLoss()

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


در این بخش مدل آموزش داده می شود.    
به این صورت که اولاون لیست امتیاز برای پیدا کردن مقدار تابغ زیان بیرون داده میشه (بغد از عمل رفت) بعدش با عملیات برگشت این ها بروزرشانی میشن و در نهایت بعد از هر گردش این مقدار ضیان بیرون داده میشه

In [25]:
model.train()
for epoch in range(EPOCHS):
    total_loss = 0.0
    for batch in data_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask)

        loss = criterion(logits, labels)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(data_loader)
    print(f"Epoch {epoch + 1}/{EPOCHS} - Loss: {avg_loss:.4f}")

Epoch 1/2 - Loss: 0.7524
Epoch 2/2 - Loss: 0.5919


# Prediction

In [ ]:
prediction = model.predict_dataframe(validation, "reviewText")
prediction_train = model.predict_dataframe(train, "reviewText")


In [ ]:
prediction.to_csv("submission.csv", index=False)

In [ ]:
prediction

,prediction
4485,5
30953,5
50842,4
22418,5
2777,1
...,...
26642,5
59722,1
20414,5
40452,5


In [ ]:
  from sklearn.metrics import f1_score


f1_validation = f1_score(validation_y, prediction, average="macro")
f1_train = f1_score(train_y, prediction_train, average="macro")

print(f"F1 Score Validation (micro): {f1_validation:.4f}")
print(f"F1 Score train (macro): {f1_train:.4f}")


F1 Score Validation (micro): 0.6221
F1 Score train (macro): 0.6852
